In [13]:
NUM_WORKERS_FOR_LOADER = 2

In [18]:
import os
from functools import partial
from typing import Literal

import datasets
import polars as pl
import torch
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset


class YambdaDataset(Dataset):
    DEFAULT_PATH = './datasets/yambda_likes_dataset'
    SECONDS_IN_DAY = 24 * 60 * 60

    def __init__(self,
                 path: str | None = None,
                 dataset_type: Literal['50m', '500m', '5b'] = '50m',
                 overwrite: bool = False,
                 mode: Literal['train', 'val'] = 'train',
                 max_seq_len: int = 256):

        self.path = path if path is not None else YambdaDataset.DEFAULT_PATH
        self.path += dataset_type

        os.makedirs(os.path.dirname(self.path), exist_ok=True)

        self.dataset = None

        if overwrite or not os.path.exists(self.path):
            self.dataset = load_dataset("yandex/yambda", data_dir=f"flat/{dataset_type}", data_files="likes.parquet")
            self.dataset : datasets.Dataset
            self.dataset = self.dataset['train'].to_polars()
            self.dataset.write_parquet(self.path)
        else:
            self.dataset = pl.read_parquet(self.path)

        self.dataset = self.dataset.with_columns(pl.col('item_id').rank("dense"))

        self.pad_id = 0
        self.num_tokens = self.dataset.max()['item_id'].item() + 1

        sep = self.dataset.max()['timestamp'] - self.SECONDS_IN_DAY * 7

        if mode == 'train':
            self.dataset = self.dataset.filter(pl.col('timestamp') <= sep)
        else:
            self.dataset = self.dataset.filter(pl.col('timestamp') > sep)

        self.dataset = self.dataset.sort('timestamp')
        self.dataset = (
            self
            .dataset.group_by('uid', maintain_order=True)
            .agg(pl.col('item_id').tail(max_seq_len))
        )
        self.dataset: pl.DataFrame
        self.dataset = [torch.LongTensor(i) for i in self.dataset['item_id'].to_list()]

    def __getitem__(self, idx) -> torch.Tensor:
        return self.dataset[idx]

    def __len__(self) -> int:
        return len(self.dataset)


class Utils:

    @staticmethod
    def collate_to_batch(batch, pad_id, max_len):
        batch_t = pad_sequence(batch, batch_first=True, padding_value=pad_id)
        batch_t = batch_t.long()
        if batch_t.shape[1] < max_len:
            shape = list(batch_t.shape)
            shape[1] = max_len - shape[1]
            batch_t = torch.concat((batch_t,
                                    torch.ones(*shape, dtype=torch.long) * pad_id), 1)
        return batch_t

    @staticmethod
    def collate_with_random_negatives(batch, pad_id, max_token, num_neg, max_len):
        batch_t = Utils.collate_to_batch(batch, pad_id, max_len)
        neg = torch.randint(0, max_token, (*batch_t.shape, num_neg), dtype=torch.long)
        return [batch_t, neg]

    @staticmethod
    def get_train_dataloader(batch_size=32, max_len=200, num_neg=256,
                             dataset_type: Literal['50m', '500m', '5b'] = '500m',
                             num_workers=4):
        dataset = YambdaDataset(max_seq_len=max_len, mode='train', dataset_type=dataset_type)

        collate_fn = partial(
            Utils.collate_with_random_negatives,
            pad_id=dataset.pad_id,
            max_token=dataset.num_tokens,
            num_neg=num_neg,
            max_len=max_len
        )

        dataloader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True,
                                num_workers=num_workers,
                                collate_fn=collate_fn)
        return dataloader

    @staticmethod
    def get_val_dataloader(batch_size=32, max_len=200,
                           dataset_type: Literal['50m', '500m', '5b'] = '500m',
                           num_workers=4):
        dataset = YambdaDataset(max_seq_len=max_len, mode='val', dataset_type=dataset_type)

        collate_fn = partial(
            Utils.collate_to_batch,
            pad_id=dataset.pad_id,
            max_len=max_len
        )

        dataloader = DataLoader(dataset=dataset, batch_size=batch_size,
                                num_workers=num_workers,
                                collate_fn=collate_fn)
        return dataloader


__all__ = ["YambdaDataset", "Utils"]


In [19]:
import torch
from torch import nn


class SASRec(nn.Module):
    def __init__(self,
                 num_embeddings: int,
                 seq_len: int,
                 embedding_dim: int = 256,
                 num_heads: int = 4,
                 num_layers: int = 4,
                 transformer_dim=2048,
                 transformer_dropout=0.1,
                 reuse_embeddings: bool = False):
        super(SASRec, self).__init__()
        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.num_layers = num_layers

        self.input_embedding = nn.Embedding(num_embeddings=num_embeddings + 1, embedding_dim=embedding_dim)
        self.position_embedding = nn.Embedding(num_embeddings=seq_len, embedding_dim=embedding_dim)

        self.transformer = nn.TransformerEncoder(
            encoder_layer=nn.TransformerEncoderLayer(
                d_model=embedding_dim,
                nhead=num_heads,
                dim_feedforward=transformer_dim,
                dropout=transformer_dropout,
                batch_first=True
            ),
            num_layers=num_layers
        )

        self.linear = nn.Linear(in_features=embedding_dim, out_features=embedding_dim)

        if reuse_embeddings:
            self.output_embedding = self.input_embedding
        else:
            self.output_embedding = nn.Embedding(num_embeddings=num_embeddings, embedding_dim=embedding_dim)

        self.pad_id = 0

    def forward(self, x):

        device = x.device

        input_embeddings = self.input_embedding(x)

        seq_len = input_embeddings.shape[1]

        positions = (torch.arange(seq_len, dtype=torch.long, device=device)
                     .unsqueeze(0).expand(x.size(0), seq_len))
        position_embeddings = self.position_embedding(positions)

        embeddings = input_embeddings + position_embeddings

        causal_mask = torch.nn.Transformer.generate_square_subsequent_mask(seq_len, device=device)

        attention = self.transformer(embeddings,
                                     mask=causal_mask,
                                     is_causal=True)

        return self.linear(attention)


In [21]:
import matplotlib.pyplot as plt
import torch
from torch.nn.functional import binary_cross_entropy_with_logits, cross_entropy
from tqdm.notebook import tqdm


def show_metrics(metric1, metric2, label1, label2, name):
    plt.figure(num=name)
    plt.plot(range(len(metric1)), metric1, label=label1)
    plt.plot(range(len(metric2)), metric2, label=label2)
    plt.legend()
    plt.grid(True)
    plt.show()


def train_epoch(model: SASRec, train_loader, optimizer):
    model.train()
    sum_loss = 0

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for i, (positives, negatives) in enumerate(tqdm(train_loader, desc="Batch")):
        optimizer.zero_grad()
        positives, negatives = positives.to(device), negatives.to(device)

        model_input = positives[:, :-1]  # B, S, E
        positives = positives[:, 1:]
        negatives = negatives[:, 1:, :]  # B, S, N, E
        neg_embeddings = model.output_embedding(negatives)
        pos_embeddings = model.output_embedding(positives)

        output = model(model_input)
        neg_logits = torch.einsum("bse, bsne -> bsn", output, neg_embeddings)
        pos_logits = torch.einsum("bse, bse -> bs", output, pos_embeddings)

        pos_logits = pos_logits.unsqueeze(-1)

        logits = torch.cat([neg_logits, pos_logits], dim=-1)
        gt = torch.cat([torch.zeros_like(neg_logits), torch.ones_like(pos_logits)], dim=-1)

        loss = binary_cross_entropy_with_logits(logits, gt, reduction="none")

        mask = (positives != model.pad_id)
        mask = mask.unsqueeze(-1).float()

        loss = loss * mask
        loss = loss.sum() / mask.sum()

        loss.backward()
        optimizer.step()

        sum_loss += loss.item()

    return sum_loss / len(train_loader)


@torch.no_grad()
def validate_epoch(model: SASRec, val_loader):
    model.eval()

    sum_loss = 0
    correct_top1 = 0
    correct_top5 = 0
    total = 0

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    all_embeddings = model.output_embedding.weight  # V, E

    for labels in tqdm(val_loader, desc="Batch"):
        labels = labels.to(device)

        model_input = labels[:, :-1]
        labels = labels[:, 1:]

        model_output = model(model_input)  # B, S, E
        logits = torch.einsum("bse, ve -> bsv", model_output, all_embeddings)

        logits = logits.reshape(-1, logits.size(-1))
        labels = labels.reshape(-1)

        loss = cross_entropy(logits, labels)
        sum_loss += loss.item()

        mask = (labels != model.pad_id).float()
        total += mask.sum().item()

        top1 = logits.argmax(dim=-1)
        top5 = logits.topk(5, dim=-1).indices

        correct_top1 += ((top1 == labels) * mask).sum().item()
        correct_top5 += (((top5 == labels.unsqueeze(-1)).any(dim=-1)) * mask).sum().item()

    avg_loss = sum_loss / len(val_loader)
    top1_acc = correct_top1 / total
    top5_acc = correct_top5 / total

    return avg_loss, top1_acc, top5_acc


def train(model: SASRec, train_loader, val_loader, optimizer, epochs):
    train_losses, val_losses = [], []
    top1_accs = []
    top5_accs = []

    for epoch in tqdm(range(epochs)):
        sum_loss, top1_acc, top5_acc = validate_epoch(model, val_loader)

        val_losses.append(sum_loss)
        top1_accs.append(top1_acc)
        top5_accs.append(top5_acc)

        train_losses.append(train_epoch(model, train_loader, optimizer))

        print(f"Epoch {epoch}: train_loss={train_losses[-1]:.4f}, val_loss={val_losses[-1]:.4f}, "
              f"top1={top1_acc:.4f}, top5={top5_acc:.4f}")

        show_metrics(train_losses, val_losses, "train", "val", "losses")
        show_metrics(top1_accs, top5_accs, "top-1", "top-5", "accuracy")


In [22]:
dataset_type = '50m'

In [25]:
%rm ./datasets -rf

In [27]:
train_loader = Utils.get_train_dataloader(dataset_type=dataset_type, num_workers=2)
val_loader = Utils.get_val_dataloader(dataset_type=dataset_type, num_workers=2)

In [28]:
dataset = YambdaDataset(dataset_type=dataset_type)

In [30]:
model = SASRec(num_embeddings=dataset.num_tokens + 1, seq_len=200)

In [31]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
device

device(type='cuda')

In [33]:
optimizer = torch.optim.Adam(model.parameters())

train(model, train_loader, val_loader, optimizer, 20)

  0%|          | 0/20 [00:00<?, ?it/s]

Batch:   0%|          | 0/125 [00:00<?, ?it/s]

KeyboardInterrupt: 